# 多模态 RAG（Captioning 版）：把图片“转成可检索的文本”

很多 PDF（论文/报告）里，关键信息并不只在正文文字里，还在：

- 图（架构图、流程图）
- 表（对比表、实验结果）

但大多数向量检索默认只对文本做 embedding。**Captioning 版多模态 RAG** 的核心做法是：

1. 从 PDF 抽出图片
2. 用多模态模型给每张图片生成“可检索摘要”（caption）
3. 把这些 caption 当作普通文本入库（embedding + vector store）
4. 问答时通过检索把“图片信息”也带回上下文

下面我们用论文《Attention Is All You Need》做演示。

## 流程图

<img src="../images/multi_model_rag_with_captioning.svg" alt="multi_model_rag_with_captioning" width="320">

## 1) 导入依赖

我们会用：

- `fitz`（PyMuPDF）：从 PDF 抽取图片 + 文本
- `PIL`：把抽出的图片保存为 PNG
- `ChatOpenAI`：对图片做 caption（多模态输入）
- `OpenAIEmbeddings + FAISS`：把文本块与图片 caption 一起入库检索

In [1]:
import base64
import io
import os
import sys
from pathlib import Path
from typing import List, Tuple

import fitz  # PyMuPDF
from PIL import Image
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_97905/4194370131.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 2) 下载示例 PDF（Attention Is All You Need）

我们把 PDF 下载到 `../data/`，避免重复下载。

In [2]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = DATA_DIR / "attention_is_all_you_need.pdf"

if not PDF_PATH.exists():
    resp = requests.get(PDF_URL, timeout=120, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    PDF_PATH.write_bytes(resp.content)

print(PDF_PATH)
print("PDF size (MB):", round(PDF_PATH.stat().st_size / 1024 / 1024, 2))

../data/attention_is_all_you_need.pdf
PDF size (MB): 2.11


## 3) 从 PDF 抽取文本与图片

我们做两件事：

- **文本**：按页抽取（后面会切成 chunks）
- **图片**：把每页里的嵌入图片抽出来保存为 PNG，并记录它来自第几页

接下来我们会对每张图片做 caption，把“视觉信息”转成可检索的文本。

In [4]:
IMG_DIR = DATA_DIR / "attention_images"
IMG_DIR.mkdir(parents=True, exist_ok=True)

text_pages: List[Document] = []
image_files: List[dict] = []  # {path, page, img_index}

with fitz.open(str(PDF_PATH)) as pdf:
    for page_i in range(len(pdf)):
        page = pdf[page_i]

        # 1) 文本（按页）
        txt = (page.get_text() or "").strip()
        if txt:
            text_pages.append(
                Document(
                    page_content=txt,
                    metadata={"type": "PDF_PAGE_TEXT", "page": page_i, "source": str(PDF_PATH)},
                )
            )

        # 2) 图片
        images = page.get_images(full=True)
        for img_i, img in enumerate(images, start=0):
            xref = img[0]
            base = pdf.extract_image(xref)
            img_bytes = base.get("image")
            if not img_bytes:
                continue

            pil = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            out_path = IMG_DIR / f"page_{page_i:03d}_img_{img_i:02d}.png"
            pil.save(out_path)

            image_files.append({"path": str(out_path), "page": page_i, "img_index": img_i})

print("Text pages:", len(text_pages))
print("Extracted images:", len(image_files))
print("Example image:", image_files[0] if image_files else None)

Text pages: 15
Extracted images: 3
Example image: {'path': '../data/attention_images/page_002_img_00.png', 'page': 2, 'img_index': 0}


## 4) 图片 Captioning：用多模态模型把图片变成“可检索摘要”

这里我们用 LangChain 的**标准多模态 content blocks** 给 `ChatOpenAI` 传入图片（base64）。

- 输入：图片 + 指令（让模型输出适合检索的短摘要）
- 输出：caption 文本（用于 embedding 与检索）

为了控制成本，我们默认只对前 `MAX_IMAGES` 张图片做 caption。

In [5]:
caption_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

caption_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是检索用图片摘要器。任务：把给定图片（可能是图/表/公式/架构图）总结成一段适合检索的中文摘要。\n"
            "要求：\n"
            "- 优先描述图片中最关键的信息点（结构、模块关系、变量含义、表格结论）\n"
            "- 不要套话，不要解释你在做什么\n"
            "- 1-4 句即可\n",
        ),
        (
            "human",
            [
                {"type": "text", "text": "请为这张图片生成检索摘要："},
                {"type": "image", "base64": "{image_b64}", "mime_type": "image/png"},
            ],
        ),
    ]
)

caption_chain = caption_prompt | caption_llm | StrOutputParser()


def image_to_b64(path: str) -> str:
    data = Path(path).read_bytes()
    return base64.b64encode(data).decode("utf-8")


MAX_IMAGES = 12

image_caption_docs: List[Document] = []
for item in image_files[:MAX_IMAGES]:
    b64 = image_to_b64(item["path"])
    cap = caption_chain.invoke({"image_b64": b64}).strip()
    if not cap:
        continue
    image_caption_docs.append(
        Document(
            page_content=cap,
            metadata={
                "type": "IMAGE_CAPTION",
                "image_path": item["path"],
                "page": int(item["page"]),
                "source": str(PDF_PATH),
            },
        )
    )

print("Captioned images:", len(image_caption_docs))
print("Example caption:\n", image_caption_docs[0].page_content if image_caption_docs else None)

Captioned images: 3
Example caption:
 该图展示了一个基于Transformer架构的模型结构，包括输入和输出的嵌入层、位置编码、多个编码器层（包含多头注意力和前馈网络）以及输出概率的计算过程。每个编码器层通过“Add & Norm”模块连接，右侧的输出层使用线性变换和Softmax函数生成最终概率。


## 5) 入库：把“文本 chunks + 图片 captions”一起向量化并存到 FAISS

关键点：

- 文本：我们把按页文本切成 chunks
- 图片：我们只存 caption（它是文本），并保留 `image_path/page` 元数据

这样检索时，**图片信息会以 caption 的形式被召回**。

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True,
)

text_chunks = splitter.split_documents(text_pages)
for i, d in enumerate(text_chunks):
    d.metadata.update({"chunk_id": i, "type": "PDF_TEXT_CHUNK"})

all_docs = text_chunks + image_caption_docs

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

VSTORE_DIR = Path("../vector_stores")
VSTORE_DIR.mkdir(parents=True, exist_ok=True)
STORE_DIR = VSTORE_DIR / "multimodal_captioning_faiss"

if STORE_DIR.is_dir():
    vectorstore = FAISS.load_local(
        str(STORE_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    vectorstore = FAISS.from_documents(all_docs, embeddings)
    vectorstore.save_local(str(STORE_DIR))

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
print("Text chunks:", len(text_chunks))
print("Caption docs:", len(image_caption_docs))
print("Vectorstore size:", int(vectorstore.index.ntotal))

Text chunks: 63
Caption docs: 3
Vectorstore size: 66


## 6) 检索：看看是否能把“图里的信息”一起召回

我们用一个更偏向图表/架构的问题来检索，然后打印返回的文档类型：

- `PDF_TEXT_CHUNK`
- `IMAGE_CAPTION`（如果召回到了，说明 caption 入库生效）

In [7]:
query = "Transformer architecture diagram: what are the main components and how do they connect?"

hits = retriever.invoke(query)
for i, d in enumerate(hits, start=1):
    print(f"[{i}] type={d.metadata.get('type')} page={d.metadata.get('page')}")
    if d.metadata.get("type") == "IMAGE_CAPTION":
        print(" image:", Path(d.metadata["image_path"]).name)
    print(d.page_content[:260].replace("\n", " "))
    print("---")

[1] type=PDF_TEXT_CHUNK page=2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively
---
[2] type=IMAGE_CAPTION page=2
 image: page_002_img_00.png
该图展示了一个基于Transformer架构的模型结构，包括输入和输出的嵌入层、位置编码、多个编码器层（包含多头注意力和前馈网络）以及输出概率的计算过程。每个编码器层通过“Add & Norm”模块连接，右侧的输出层使用线性变换和Softmax函数生成最终概率。
---
[3] type=PDF_TEXT_CHUNK page=1
End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [34]. To the best of our knowledge, however, t
---
[4] type=PDF_TEXT_CHUNK page=1
tion models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]. In all but a few cases [27], however, such a

## 7) 问答：用检索到的上下文回答问题

这里先用“caption 文本 + 文本 chunk”拼接成 context 做问答。

如果你想进一步做真正的多模态 RAG，也可以在回答阶段把 `IMAGE_CAPTION` 对应的图片以 content blocks 形式再喂给模型。

In [8]:
qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个严谨的问答助手。只能使用给定的上下文回答问题。\n"
            "如果上下文不包含答案，直接回答：我不知道。",
        ),
        ("human", "问题：{question}\n\n上下文：\n{context}\n\n回答："),
    ]
)
qa_chain = qa_prompt | qa_llm | StrOutputParser()


def docs_to_context(docs: List[Document]) -> str:
    parts = []
    for d in docs:
        t = d.metadata.get("type")
        if t == "IMAGE_CAPTION":
            parts.append(f"[IMAGE_CAPTION page={d.metadata.get('page')}] {d.page_content}")
        else:
            parts.append(d.page_content)
    return "\n\n".join(parts)

context = docs_to_context(hits)
answer = qa_chain.invoke({"question": query, "context": context}).strip()
print(answer)

Transformer架构的主要组件包括编码器和解码器堆栈。编码器由N=6个相同的层组成，每层有两个子层：多头自注意力机制和位置逐点全连接前馈网络。每个子层周围都有残差连接，随后进行层归一化。解码器的结构与编码器类似，但在生成输出序列时是自回归的。编码器将输入序列映射为连续表示，解码器则基于这些表示生成输出序列。各个编码器层通过“Add & Norm”模块连接，最终输出层使用线性变换和Softmax函数生成概率。
